# POSE — `infer.ipynb` · Bild rein → 3D-Viewer raus

Dieses Notebook ist ein duenner Client fuer den **KIP-Pose-Webservice** auf der
GPU-Workstation. Ein Bild geht rein, ein schema-valides `pose_result.json` kommt
raus, und der Three.js-Viewer auf `max-utils.com/KIP` zeigt die geschaetzten 6D-
Posen auf dem CAD der Maschinenzelle.

## Pipeline (ADR-018)

```
Bild (input/) ─Detektor(YOLOv8-OBB)─▶ OBB→AABB-Crops
              ─GDRNPP(RGB-only, per-Objekt, warm im VRAM)─▶ (R_m2c, t_m2c) [BOP, mm, Kamera-Frame]
              ─bop_adapter §3 (Welt-Transform + Boden-Snap)─▶ (R_world, t_world, upright)
              ─▶ pose_result.json  (Contract: pose_result.schema.json)
frontend/     pose_result.json + cell.glb ─▶ Three.js-Viewer (echtes CAD @ 6D-Pose)
```

## Produktions-Pipeline

Alle drei trainierten Objekte (Anker_Kurz AR 0.870, Anker_Lang 0.907, Zahnrad 0.838)
laufen ueber den persistenten `kip-worker.service` auf Port 8078 — drei Modelle
`model_best.pth` warm im VRAM, ein Foto kostet ~4s end-to-end (Detektor + GDRNPP +
Welt-Transform + Snap). Kein Checkpoint-Pfad noetig — das Notebook fragt nur den
Webservice.

> RGB-only **hart** (ADR-018). Konvention: Z-up Welt, `world = R @ body`,
> Ursprung = Tisch-Nullpunkt, Einheit Meter.

**Lokale Abhaengigkeiten:** `requests`, `Pillow` (Detektor-Overlay), `numpy`.
Kein lokales Torch/Ultralytics noetig — alles laeuft auf der Workstation.

## 1 · Eingabebild laden (aus `project/input/`)

Lege dein Szenenbild nach `project/input/` (z.B. `scene_0000.png`). Liegt eine
`bbox_2d_*.json` (SDG-Annotator-Format) oder `scene_camera.json` daneben,
werden sie automatisch genutzt. Ist `input/` leer, erzeugt die nächste Zelle
ein synthetisches Demo-Bild, damit das Notebook **ohne manuelles Setup** läuft.

In [ ]:
import os, pathlib, json, requests
from PIL import Image
from io import BytesIO

# KIP-Pose-Webservice (warmer GDRNPP-Worker auf der GPU-Workstation).
WEBSERVICE_URL = os.environ.get("KIP_WEBSERVICE_URL",
                                "https://max-utils.com/KIP")
# Intern (schneller, nur im Tailnet): KIP_WEBSERVICE_URL=http://<gpu-host>:8077

PROJECT = pathlib.Path.cwd()
if PROJECT.name != 'project': PROJECT = PROJECT / 'project'
INPUT_DIR = PROJECT / 'input'; TEMP_DIR = PROJECT / 'temp'
INPUT_DIR.mkdir(exist_ok=True); TEMP_DIR.mkdir(exist_ok=True)

# Health-Check: laeuft der Webservice + sind Modelle warm?
h = requests.get(f"{WEBSERVICE_URL}/api/health", timeout=10).json()
print(f"Webservice: {h['status']} | trainierte Objekte: {h['trained_objects']} | "
      f"Training aktiv: {h['gpu_training_active']}")


In [ ]:
def find_input_image():
    """Erstes Bild in input/ — sonst ein synthetisches Demo-Bild erzeugen."""
    imgs = sorted([p for p in INPUT_DIR.glob('*')
                   if p.suffix.lower() in ('.png', '.jpg', '.jpeg')])
    if imgs:
        return imgs[0], False
    # Synthetisches Demo: grauer Hintergrund + 3 farbige Teil-Blobs + bbox-JSON.
    W, H = 1280, 720
    rng = np.random.default_rng(7)
    arr = rng.integers(45, 80, size=(H, W, 3)).astype(np.uint8)
    demo = Image.fromarray(arr); d = ImageDraw.Draw(demo)
    blobs = [((300, 200, 360, 480), (210, 90, 90), 'anker_kurz', 0),
             ((700, 150, 760, 470), (90, 160, 210), 'anker_lang', 1),
             ((540, 320, 640, 420), (120, 200, 120), 'zahnrad',    5)]
    rows = []
    for (x0, y0, x1, y1), col, _, cls in blobs:
        d.ellipse([x0, y0, x1, y1], fill=col)
        rows.append([cls, x0, y0, x1, y1, 0.0])
    img_path = INPUT_DIR / 'scene_demo.png'; demo.save(img_path)
    bbox_doc = {'data': rows, 'info': {'idToLabels': {
        '0': {'class': 'anker_kurz'}, '1': {'class': 'anker_lang'},
        '5': {'class': 'zahnrad'}}}}
    json.dump(bbox_doc, open(INPUT_DIR / 'bbox_2d_demo.json', 'w'))
    return img_path, True

IMAGE, is_demo = find_input_image()
print(('DEMO-Bild erzeugt: ' if is_demo else 'Eingabebild: ') + str(IMAGE))
Image.open(IMAGE).convert('RGB')

## 2 · Detektion + GDRNPP-Inferenz via Webservice

Der Webservice kapselt beides in einem Call: YOLOv8-OBB-Detektor (3 Anker / Zahnrad) +
warmer GDRNPP-Worker (Modell pro Objekt im VRAM) liefern in ~4s das fertige
`pose_result` (3D-Posen im pose_frame, Boden-gesnappt).

In [ ]:
# Bild an den Webservice schicken: Detektor (YOLOv8-OBB) + GDRNPP-Worker
# liefern in EINEM Call die fertigen 3D-Posen im pose_result-Format.
with open(IMAGE, 'rb') as f:
    r = requests.post(f"{WEBSERVICE_URL}/api/real/infer",
                      files={"image": f}, timeout=120)
r.raise_for_status()
resp = r.json()
print(f"Detektionen: {resp['n_det']}  |  3D-Posen: {resp['n_parts']}  |  Job: {resp['job']}")

# Pose-Result-Doc abholen.
doc = requests.get(f"{WEBSERVICE_URL}/api/real/result/{resp['job']}", timeout=20).json()


In [ ]:
# Detektionen aufs Bild zeichnen (PIL — immer verfügbar).
def draw_detections(rgb, dets):
    im = Image.fromarray(rgb).convert('RGB'); dr = ImageDraw.Draw(im)
    for d in dets:
        x0, y0, x1, y1 = d['bbox_2d']
        dr.rectangle([x0, y0, x1, y1], outline=(255, 80, 80), width=3)
        dr.text((x0 + 3, max(0, y0 - 14)), f"{d['instance_id']}:{d['part']}",
                fill=(255, 220, 0))
    return im

det_vis = draw_detections(rgb, dets)
det_vis.save(TEMP_DIR / 'detections.png')
print('Overlay ->', TEMP_DIR / 'detections.png')
det_vis

## 3 · Welche Modelle sind warm?

Der Worker laedt `model_best.pth` pro trainiertem Objekt einmal ins VRAM und
bedient on-demand. `/api/health` listet die geladenen Objekte.

In [ ]:
# Hinweis: die trainierten GDRNPP-Modelle (model_best.pth pro Objekt) sind
# warm im Worker (Port 8078 auf der Workstation). Der /api/real/infer-Call oben
# nutzt sie on-demand (~4s pro Bild). Lokale Checkpoint-Pfade nicht mehr noetig.
print("Worker-Modelle:", h["trained_objects"])


## 4 · `pose_result.json` anzeigen

Der Webservice gibt das fertige Contract-Dokument zurueck (Z-up Welt, Posen
im pose_frame relativ zum Tisch-Nullpunkt). Wir sichern es lokal in `temp/`.

In [ ]:
# pose_result lokal sichern (optional) und anzeigen.
OUT = TEMP_DIR / f"pose_result_{resp['job']}.json"
json.dump(doc, open(OUT, "w"), indent=2)
print(f"pose_result -> {OUT.relative_to(PROJECT.parent)}\n")

print(f"{'#':>2}  {'part':<22} {'face':<10} {'conf':>5}  {'t_world (m)':<26} upright")
print("-"*84)
for r_ in doc["results"]:
    t = "[" + ", ".join(f"{x:+.3f}" for x in r_["t_world"]) + "]"
    print(f"{r_['instance_id']:>2}  {r_['part']:<22} {r_.get('face','-'):<10} "
          f"{r_['confidence']:>5.2f}  {t:<26} {r_['upright']}")


In [ ]:
# pose_result-Tabelle (eine Zeile pro Teil).
def show_pose_table(doc):
    print(f"{'#':>2}  {'part':<22} {'face':<10} {'conf':>5}  {'t_world (m)':<24} upright")
    print('-' * 78)
    for r in doc['results']:
        t = '[' + ', '.join(f'{v:+.3f}' for v in r['t_world']) + ']'
        print(f"{r['instance_id']:>2}  {r['part']:<22} {r['face']:<10} "
              f"{r['confidence']:>5.2f}  {t:<24} {r['upright']}")

show_pose_table(doc)
print('\nmeta:', json.dumps(doc['meta'], indent=2))

## 5 · (Optional) BOP-Eval gegen den Holdout-Split

Die offizielle Quantifizierung uebernimmt `box_src/eval_bop.py` mit den
`bop_toolkit_lib`-Metriken (AR = MSSD/MSPD, ADD/ADI, alle symmetrie-bewusst).
Headline ist `AR = mean(AR_MSSD, AR_MSPD)` — final auf dem Val-Split:
Anker_Kurz **0.870**, Anker_Lang **0.907**, Zahnrad **0.838**, Mittel **0.872**.

Eval laeuft auf der Box (`bop_toolkit_lib` ist dort installiert). Vom Laptop
via Wrapper:

```bash
# Selbst-Test ohne Checkpoint (beweist nur die Metrik-Mechanik):
box_src/eval_bop.sh --self-test

# Bestehende preds_best.csv eines Objekts neu scoren:
box_src/eval_bop.sh --preds /mnt/data/bop/repos/gdrnpp/output/gdrn/poseIsaacPbrSO/anker_kurz/preds_best.csv
```

Der Wrapper holt `report.json`/`report.txt` nach `./results/eval/`. Dieses
Inferenz-Notebook erzeugt **eine** Szene fuer den Viewer; die Quantifizierung
ueber den ganzen Val-Split macht der Eval-Harness.

## 6 · 3D-Viewer (`max-utils.com/KIP`)

Der Three.js-Viewer ist Teil des Webservices. Per Browser unter
`https://max-utils.com/KIP/` aufrufen — die linke Card zeigt die 3D-Posen
(rot) auf der Maschinenzelle, die PiP unten-rechts das Foto mit Detektor-Boxen.

In [ ]:
from IPython.display import IFrame, display, HTML

# Der 3D-Viewer ist Teil des Webservices. Bilder per Browser hochladen, oder
# das eben inferierte Job-Result direkt anzeigen.
VIEWER_URL = "https://max-utils.com/KIP/"     # public via CF-Tunnel
# VIEWER_URL = "http://<gpu-host>:8077/"   # intern (Tailscale)

print(f"3D-Viewer: {VIEWER_URL}")
display(HTML(f'<a href="{VIEWER_URL}" target="_blank">Viewer in neuem Tab oeffnen</a>'))


In [ ]:
# Viewer eingebettet (falls IFrame-Sandbox es erlaubt).
IFrame(VIEWER_URL, width='100%', height=720)


### Alternativer Start (ohne Notebook)

Ein einziger Befehl vom Repo-Root — erzeugt `pose_result` **und** öffnet den Viewer:

```bash
python project/e2e_infer.py --image project/input/scene_demo.png --serve
```

Mit echtem Checkpoint:

```bash
python project/e2e_infer.py --image project/input/<bild>.png \
    --checkpoint /pfad/zu/gdrnpp_model.pth --serve
```